[//]: # (cr:doc name='chapter_8_baseline_experiments' id=1cac8e52)
# Chapter 8: Baseline Experiments

**Purpose:** Train baseline models to understand data predictability and establish performance benchmarks.

**What you'll learn:**
- How to prepare data for ML with proper train/test splitting
- How to handle class imbalance with class weights
- How to evaluate models with appropriate metrics (not just accuracy!)
- How to interpret feature importance

**Outputs:**
- Baseline model performance (AUC, Precision, Recall, F1)
- Feature importance rankings
- ROC and Precision-Recall curves
- Performance benchmarks for comparison

---

## Evaluation Metrics for Imbalanced Data

| Metric | What It Measures | When to Use |
|--------|-----------------|-------------|
| **AUC-ROC** | Ranking quality across thresholds | General model comparison |
| **Precision** | "Of predicted churned, how many are correct?" | When false positives are costly |
| **Recall** | "Of actual churned, how many did we catch?" | When missing churners is costly |
| **F1-Score** | Balance of precision and recall | When both matter equally |
| **PR-AUC** | Precision-Recall under curve | Better for imbalanced data |

[//]: # (cr:doc name='8_1_setup' id=baf6acf3)
## 8.1 Setup

In [ ]:
# @cr:code name='init_progress' id=d4249c73
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("08_baseline_experiments.ipynb")

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

from customer_retention.analysis.auto_explorer import ExplorationFindings
from customer_retention.analysis.visualization import ChartBuilder, display_figure, display_table
from customer_retention.core.compat import (
    bulk_label_encode,
    bulk_median_impute,
    bulk_zero_variance_cols,
    collect_for_sklearn,
    concat,
    lazy_fillna,
    native_pd,
    safe_fillna,
    spark_checkpoint,
)
from customer_retention.core.config.column_config import NON_FEATURE_COLUMN_TYPES, ColumnType
from customer_retention.core.config.experiments import (
    FINDINGS_DIR,
)
from customer_retention.core.naming import composite_name as _compute_cn
from customer_retention.stages.modeling.mlflow_logger import MLFLOW_AVAILABLE, MLflowLogger
from customer_retention.stages.temporal import TEMPORAL_METADATA_COLS

In [ ]:
# @cr:config name='cv_settings' id=88ceba53
CV_FOLDS = 5

In [ ]:
# @cr:code name='load_findings' id=9621e77a
from customer_retention.analysis.auto_explorer import load_notebook_findings, resolve_target_column
from customer_retention.analysis.auto_explorer.active_dataset_store import (
    require_silver_merged,
    require_silver_merged_distributed,
)
from customer_retention.core.compat import is_remote_spark, is_spark_available, track_stage_object

FINDINGS_PATH, _namespace, dataset_name = load_notebook_findings(
    "08_baseline_experiments.ipynb", prefer_merged=True
)
print(f"Using: {FINDINGS_PATH}")

findings = ExplorationFindings.load(FINDINGS_PATH)
target = resolve_target_column(_namespace, findings)

_use_distributed = is_spark_available() and not is_remote_spark()

# Always load from silver_merged — temporal split requires as_of_date spine
if _use_distributed:
    df = require_silver_merged_distributed(_namespace)
else:
    df = require_silver_merged(_namespace)
track_stage_object(df)
data_source = "silver_merged"

charts = ChartBuilder()

print(f"\nLoaded {len(df):,} rows from: {data_source}")
if _use_distributed:
    print("  (distributed mode — data stays on Spark until sklearn boundary)")

[//]: # (cr:doc name='8_2_prepare_data_for_modeling' id=a47b4877)
## 8.2 Prepare Data for Modeling

Transforms the merged silver feature matrix into a training-ready dataset through four stages: prerequisite validation, feature availability filtering, gold-layer transforms, entity-aware temporal splitting, and L1-regularized feature selection.

### 8.2.1 Validate Prerequisites

Confirms the target column, entity key, and temporal anchor (`as_of_date`) are present in the merged dataset. Fails fast if any are missing so that downstream errors don't mask the root cause.

In [ ]:
# @cr:code name='check_skip_modeling' id=b0c078ae
_skip_modeling = False
if not target:
    print("No target column set. Skipping baseline experiments.")
    _skip_modeling = True
elif target not in df.columns:
    print(f"Target column '{target}' not found in loaded data.")
    print("Skipping baseline experiments.")
    _skip_modeling = True
elif "as_of_date" not in df.columns:
    raise ValueError("Column 'as_of_date' missing from silver_merged. Re-run notebook 03.")
elif "entity_id" not in df.columns:
    raise ValueError("Column 'entity_id' missing from silver_merged. Re-run notebook 03.")
elif len(df) == 0:
    print("No data rows loaded. Skipping baseline experiments.")
    _skip_modeling = True
elif not df[target].notna().any():
    raise ValueError(
        f"Target column '{target}' is entirely null in silver_merged. "
        "Verify that NB03 merge propagated the target from source dataset(s)."
    )

if not _skip_modeling:
    y = df[target]

    feature_cols = [
        name for name, col in findings.columns.items()
        if col.inferred_type not in NON_FEATURE_COLUMN_TYPES
        and name not in TEMPORAL_METADATA_COLS
    ]

    print("=" * 70)
    print("FEATURE SELECTION FROM FINDINGS")
    print("=" * 70)
    print(f"\n  Target Column: {target}")
    print(f"  Features Selected: {len(feature_cols)}")

    type_counts = {}
    for name in feature_cols:
        col_type = findings.columns[name].inferred_type.value
        type_counts[col_type] = type_counts.get(col_type, 0) + 1

    print("\n  Features by Type:")
    for col_type, count in sorted(type_counts.items()):
        print(f"   {col_type}: {count}")

    excluded = [name for name, col in findings.columns.items()
                if col.inferred_type in NON_FEATURE_COLUMN_TYPES]
    if excluded:
        print(f"\n  Excluded Columns ({len(excluded)}): {', '.join(excluded[:10])}{'...' if len(excluded) > 10 else ''}")

[//]: # (cr:doc name='8_2_2_feature_availability' id=fa03d008)
### 8.2.2 Feature Availability Check

Features that appear partway through the observation window create train/test distribution shift — the model learns from their presence in recent data but they are absent in older snapshots. This step removes features flagged with temporal coverage gaps so the model trains on consistently available signals only.

In [ ]:
# @cr:code name='prepare_features' id=4378ffe5
if not _skip_modeling:
    # Check feature availability and remove problematic features
    from customer_retention.stages.features.feature_selector import FeatureSelector

    print("=" * 70)
    print("FEATURE AVAILABILITY CHECK")
    print("=" * 70)

    unavailable_features = []
    if findings.has_availability_issues:
        selector = FeatureSelector(target_column=target)
        availability_recs = selector.get_availability_recommendations(findings.feature_availability)
        unavailable_features = [rec.column for rec in availability_recs]

        print(f"\n⚠️  {len(availability_recs)} feature(s) have availability issues:\n")
        for rec in availability_recs:
            print(f"   • {rec.column} ({rec.issue_type}, {rec.coverage_pct:.0f}% coverage)")

        print("\n📋 Alternative approaches (for investigation):")
        print("   • segment_by_cohort: Train separate models per availability period")
        print("   • add_indicator: Create availability flags and impute missing")
        print("   • filter_window: Restrict data to feature's available period")

        original_count = len(feature_cols)
        feature_cols = [f for f in feature_cols if f not in unavailable_features]

        if _namespace and availability_recs:
            import yaml as _yaml

            from customer_retention.analysis.auto_explorer.layered_recommendations import RecommendationRegistry
            _rec_path = _namespace.merged_recommendations_path
            if _rec_path.exists():
                with _rec_path.open() as _f:
                    _recs = RecommendationRegistry.from_dict(_yaml.safe_load(_f))
                for _ar in availability_recs:
                    _recs.add_gold_drop_availability(
                        _ar.column, _ar.issue_type, _ar.coverage_pct,
                        f"{_ar.issue_type} ({_ar.coverage_pct:.0f}% coverage)", "08_baseline_experiments",
                    )
                _recs.save(_rec_path)

        print(f"\n🗑️  Removed {original_count - len(feature_cols)} unavailable features")
        print(f"📊 Features remaining: {len(feature_cols)}")
    else:
        print("\n✅ All features have full temporal coverage.")
else:
    print("Skipped (no target column).")

[//]: # (cr:doc name='8_2_3_gold_transforms' id=9e09a801)
### 8.2.3 Gold-Layer Transforms

Applies the encoding, scaling, and derived-column transforms recommended by notebooks 04–06. Transforms are fitted here (on training data only) and their fitted state is persisted to the artifact store so that scoring can replay them identically — preventing train/serve skew.

In [ ]:
# @cr:code name='apply_gold_transforms' id=f8a2c4e1
if not _skip_modeling:
    import time as _time

    from customer_retention.analysis.auto_explorer.layered_recommendations import RecommendationRegistry
    from customer_retention.generators.pipeline_generator.gold_transform_applicator import (
        build_gold_steps,
        build_silver_derived_steps,
    )
    from customer_retention.transforms import TransformExecutor, print_step_progress
    from customer_retention.transforms.artifact_store import ArtifactStore

    _recs_hash = None
    _recs_path = _namespace.merged_recommendations_path if _namespace else None
    if _recs_path and _recs_path.exists():
        _registry = RecommendationRegistry.load(_recs_path)
        _recs_hash = _registry.compute_recommendations_hash()
        _executor = TransformExecutor()
        _cols = set(df.columns)
        _t0 = _time.perf_counter()

        _silver_steps = build_silver_derived_steps(_registry, _cols)
        if _silver_steps:
            _t_silver = _time.perf_counter()
            df = _executor.apply_all(df, _silver_steps, on_step_done=print_step_progress)
            feature_cols = [c for c in df.columns if c in set(feature_cols) | {s.column for s in _silver_steps}]
            print(f"Applied {len(_silver_steps)} silver derived columns ({_time.perf_counter() - _t_silver:.1f}s)")

        _pre_gold_cols = set(df.columns)
        _gold_steps = build_gold_steps(_registry, _pre_gold_cols)
        if _gold_steps:
            _t_gold = _time.perf_counter()
            _artifacts_dir = _namespace.artifacts_dir(_recs_hash)
            _store = ArtifactStore(_artifacts_dir)
            df = _executor.apply_all(df, _gold_steps, fit_mode=True, artifact_store=_store, on_step_done=print_step_progress)
            _store.save_manifest()
            _new_cols = [c for c in df.columns if c not in _pre_gold_cols]
            feature_cols = feature_cols + _new_cols
            print(f"Applied {len(_gold_steps)} gold transforms ({len(_new_cols)} new columns, {_time.perf_counter() - _t_gold:.1f}s)")
            print(f"Fitted artifacts saved to: {_artifacts_dir}")

        print(f"Transforms total: {_time.perf_counter() - _t0:.1f}s")
    else:
        print("No recommendations file found, skipping gold transforms")
else:
    print("Skipped (no target column).")

In [ ]:
# @cr:code name='apply_leakage_prefix_exclusion' id=a7c3e1f0
if not _skip_modeling and _namespace:
    from customer_retention.analysis.auto_explorer.exploration_manager import MultiDatasetFindings
    from customer_retention.generators.pipeline_generator.findings_parser import FindingsParser

    _multi_path = _namespace.multi_dataset_findings_path
    _exclusion_prefixes: list[str] = []
    if _multi_path.exists():
        _mdf = MultiDatasetFindings.load(_multi_path)
        for _ds_info in _mdf.datasets.values():
            for _col in _ds_info.excluded_leaking_features:
                _exclusion_prefixes.append(f"{_col}_")
        _exclusion_prefixes = sorted(set(_exclusion_prefixes))

    _prefix_drops = FindingsParser.find_leakage_excluded_columns(df.columns, _exclusion_prefixes)
    if _prefix_drops:
        df = df.drop(columns=_prefix_drops)
        feature_cols = [c for c in feature_cols if c not in set(_prefix_drops)]
        print(f"Dropped {len(_prefix_drops)} leakage-excluded columns (prefix match)")
    else:
        print("No columns matched leakage exclusion prefixes")

[//]: # (cr:doc name='8_2_4_temporal_split' id=b712b950)
### 8.2.4 Entity-Aware Temporal Split

Splits the data by time so that all test rows come from a later period than training rows. All snapshots of a given entity stay entirely in train or test — this prevents the model from seeing future behaviour of an entity it will be asked to predict. A configurable purge gap removes training rows temporally adjacent to the test boundary to eliminate label leakage from the transition zone.

In [ ]:
# @cr:code name='split_train_test' id=d53b5062
if not _skip_modeling:
    import os

    from customer_retention.analysis.auto_explorer.project_context import ProjectContext
    from customer_retention.core.compat import is_databricks
    from customer_retention.core.config.experiments import get_experiments_dir
    from customer_retention.stages.modeling.training_preparator import PreparationProgressTracker, TrainingPreparator

    _project_ctx = ProjectContext.load(_namespace.project_context_path) if _namespace and _namespace.project_context_path.exists() else None
    _purge_gap = (_project_ctx.intent.purge_gap_days if _project_ctx and _project_ctx.intent else None) or 104

    _source_names = sorted(_project_ctx.datasets.keys()) if _project_ctx else (_namespace.list_datasets() if _namespace else [])
    _cn = _compute_cn(_source_names) if _source_names else "unknown"
    _on_databricks = is_databricks()
    _experiment_name = f"/Shared/training_{_cn}" if _on_databricks else f"training_{_cn}"
    _tracking_uri = f"sqlite:///{get_experiments_dir() / 'mlruns.db'}" if not _on_databricks else None

    _preparator = TrainingPreparator(
        target_column=target, feature_columns=feature_cols,
        purge_gap_days=_purge_gap, max_rows=int(os.environ["CR_MAX_BASELINE_ROWS"]) if os.environ.get("CR_MAX_BASELINE_ROWS") else None,
        on_progress=PreparationProgressTracker(),
    )
    _prep = _preparator.prepare(df)

    X_train, X_test = _prep.X_train, _prep.X_test
    y_train, y_test = _prep.y_train, _prep.y_test
    X_train_scaled, X_test_scaled = _prep.X_train_scaled, _prep.X_test_scaled
    y_test_np, _feature_names = _prep.y_test_np, _prep.feature_names
    _train_entities, _train_dates = _prep.train_entities, _prep.train_dates
    _split_method = "temporal (purge gap)"
    if _use_distributed:
        train_count, test_count = _prep.split_info["train_size"], _prep.split_info["test_size"]
    del df

    if _namespace and _prep.zero_variance_dropped:
        import yaml as _yaml

        from customer_retention.analysis.auto_explorer.layered_recommendations import RecommendationRegistry
        _rec_path = _namespace.merged_recommendations_path
        if _rec_path.exists():
            with _rec_path.open() as _f:
                _recs = RecommendationRegistry.from_dict(_yaml.safe_load(_f))
            for _zv_col in _prep.zero_variance_dropped:
                _recs.add_gold_drop_zero_variance(_zv_col, 0.0, "Zero variance in training data", "08_baseline_experiments")
            _recs.save(_rec_path)

    print(f"Purge gap: {_purge_gap} days | Cutoff: {_prep.split_info.get('cutoff_date', 'N/A')}")
    print(f"Purged: {_prep.split_info.get('purge_gap_rows', 0)} | Zero-var dropped: {len(_prep.zero_variance_dropped)}")
    print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
    _vc = _prep.class_distribution
    _train_total = sum(_vc.values())
    _retained, _churned = int(_vc.get(1, 0)), int(_vc.get(0, 0))
    _is_binary = len(_vc) == 2
    print(f"Retained (1): {_retained:,} ({_retained/_train_total*100:.1f}%) | Churned (0): {_churned:,} ({_churned/_train_total*100:.1f}%)")

[//]: # (cr:doc name='8_2_5_feature_selection' id=c284863a)
### 8.2.5 Feature Selection

Two-stage reduction of the feature set:

1. **NB05 statistical drops** — features flagged as near-constant (`drop_weak`) or pairwise-redundant (`drop_multicollinear`) are removed first. These are data-intrinsic filters that don't need a target split.

2. **L1-regularised logistic regression** — fits a penalised model on the **training split only** (no test leakage). The L1 penalty drives coefficients of non-predictive features to exactly zero; those features are dropped. This catches features that passed statistical filters individually but add no marginal value in the presence of other features.

**Key parameters:**

- **`L1_REGULARIZATION_C`** — inverse regularization strength. *Lower* values apply stronger penalty (more features zeroed out). *Higher* values are more permissive. `C=1.0` is aggressive and may over-prune; `C=10.0` is a moderate starting point; `C=100.0` is lenient and keeps most features that carry any signal.

- **`L1_RATIO`** — controls the L1/L2 mix. `1.0` = pure Lasso, which aggressively zeros out individual features. Values below 1.0 blend in L2 (Ridge) regularisation — this is called **ElasticNet**. ElasticNet is useful when features are correlated in groups (e.g., multiple aggregation windows of the same metric): pure L1 arbitrarily picks one from each group and kills the rest, while ElasticNet (`0.5`–`0.7`) tends to keep or drop the group together.

**Tuning guide:**

| Symptom | Adjustment |
|---------|------------|
| AUC ≈ 0.5 (no signal left) | Increase `C` (try `100.0`) or lower `L1_RATIO` (try `0.7`) |
| Too many features survive, training is slow | Decrease `C` (try `1.0`) |
| Correlated feature groups lose all but one member | Lower `L1_RATIO` to `0.5`–`0.7` (ElasticNet) |
| Want to skip L1 entirely and keep all features | Set `L1_FEATURE_SELECTION_ENABLED = False` |

In [ ]:
# @cr:config name='l1_selection_config' id=35b1a3db
L1_FEATURE_SELECTION_ENABLED = True
FEATURE_SELECTION_MAX_FEATURES = 500
APPLY_NB05_DROPS = True
L1_REGULARIZATION_C = 10.0   # inverse penalty: 1.0=aggressive, 10.0=moderate, 100.0=permissive
L1_RATIO = 1.0               # 1.0=pure Lasso, 0.5-0.7=ElasticNet (keeps correlated groups)

In [ ]:
# @cr:code name='run_l1_feature_selection' id=33fccd09
if not _skip_modeling:
    from customer_retention.stages.modeling.cross_validator import _CV_DATE_COL, _CV_ENTITY_COL
    _CV_META = {_CV_ENTITY_COL, _CV_DATE_COL}
    _pre_count = sum(1 for c in X_train.columns if c not in _CV_META)
    _nb05_drops = set()

    if APPLY_NB05_DROPS and _namespace:
        _rec_path = _namespace.merged_recommendations_path
        if _rec_path.exists():
            import yaml as _yaml

            from customer_retention.analysis.auto_explorer.layered_recommendations import RecommendationRegistry
            with _rec_path.open() as _f:
                _recs = RecommendationRegistry.from_dict(_yaml.safe_load(_f))
            if hasattr(_recs, 'gold') and _recs.gold:
                _nb05_drops = {
                    r.target_column for r in getattr(_recs.gold, 'feature_selection', [])
                    if r.action in ('drop_multicollinear', 'drop_weak')
                }
        if _nb05_drops:
            _to_drop = [c for c in _nb05_drops if c in X_train.columns]
            X_train = X_train.drop(columns=_to_drop)
            X_test = X_test.drop(columns=_to_drop)
            X_train_scaled = X_train_scaled.drop(columns=[c for c in _to_drop if c in X_train_scaled.columns])
            X_test_scaled = X_test_scaled.drop(columns=[c for c in _to_drop if c in X_test_scaled.columns])
            print(f"Pre-filtered {len(_to_drop)} features from NB05 recommendations")

    _l1_drops: set = set()
    _l1_reasons: dict = {}
    _l1_scores: dict = {}
    if L1_FEATURE_SELECTION_ENABLED:
        from customer_retention.core.compat import concat
        from customer_retention.stages.features.feature_selector import run_selection_pipeline

        _l1_features = [c for c in X_train.columns if c not in _CV_META]
        _sel_df = concat([X_train[_l1_features + [_CV_DATE_COL]], y_train.rename(target)], axis=1)
        _sel_result = run_selection_pipeline(
            _sel_df, target_column=target,
            variance_threshold=0.0, correlation_threshold=1.0,
            l1_enabled=True, max_features=FEATURE_SELECTION_MAX_FEATURES,
            l1_C=L1_REGULARIZATION_C, l1_ratio=L1_RATIO,
            temporal_column=_CV_DATE_COL,
        )
        _l1_drops = set(_sel_result.dropped_features)
        _l1_reasons = _sel_result.drop_reasons
        _l1_scores = _sel_result.importance_scores or {}

        if _l1_drops:
            _keep = [c for c in X_train.columns if c not in _l1_drops]
            X_train = X_train[_keep]
            X_test = X_test[[c for c in _keep if c in X_test.columns]]
            _keep_scaled = [c for c in _keep if c in X_train_scaled.columns]
            X_train_scaled = X_train_scaled[_keep_scaled]
            X_test_scaled = X_test_scaled[[c for c in _keep_scaled if c in X_test_scaled.columns]]

        if _namespace and _l1_drops:
            _rec_path = _namespace.merged_recommendations_path
            if _rec_path.exists():
                import yaml as _yaml

                from customer_retention.analysis.auto_explorer.layered_recommendations import RecommendationRegistry
                with _rec_path.open() as _f:
                    _recs = RecommendationRegistry.from_dict(_yaml.safe_load(_f))
                for _feat, _reason in _l1_reasons.items():
                    _coeff = _l1_scores.get(_feat, 0.0)
                    _recs.add_gold_drop_l1_zero(_feat, _coeff, _reason, "08_baseline_experiments")
                _recs.save(_rec_path)

    import gc as _gc
    del _sel_df, _sel_result
    _gc.collect()
    del _gc

    _feature_names = [c for c in X_train.columns if c not in _CV_META]
    _post_count = len(_feature_names)
    print(f"\nFeature selection: {_pre_count} -> {_post_count} features")
    if _nb05_drops:
        print(f"  NB05 statistical drops: {len(_nb05_drops)}")
    if L1_FEATURE_SELECTION_ENABLED:
        print(f"  L1 drops: {len(_l1_drops)}")
        print(f"  L1 config: C={L1_REGULARIZATION_C}, l1_ratio={L1_RATIO}")

[//]: # (cr:doc name='8_2_6_feature_profile_snapshot' id=06dad7e8)
### 8.2.6 Feature Profile Snapshot

Captures a fingerprint of the post-selection feature set — column names, dtypes, null counts — together with a codified record of every excluded column and the reason it was dropped (`metadata`, `drop_multicollinear`, `drop_weak`, `drop_l1_zero`). The production pipeline saves an equivalent profile so that `compare_feature_profiles` can detect drift between exploration and production: missing/extra features, type mismatches, and selection divergence.

In [ ]:
# @cr:code name='save_exploration_feature_profile' id=2fd112fd
if not _skip_modeling and _namespace:
    from customer_retention.stages.modeling.feature_profile import ColumnProfile, build_feature_profile

    _post_sel_features = [c for c in X_train.columns if c not in _CV_META]
    _profile_row_count = _prep.split_info['train_size'] + _prep.split_info['test_size']
    _profile_stats = {
        col: ColumnProfile(dtype=str(X_train[col].dtype), non_null_count=_profile_row_count - _prep.null_counts.get(col, 0), null_count=_prep.null_counts.get(col, 0))
        for col in _post_sel_features
    }

    _excluded_cols = {col: "metadata" for col, info in findings.columns.items() if info.inferred_type in NON_FEATURE_COLUMN_TYPES}
    _excluded_cols[_CV_ENTITY_COL] = "cv_metadata"
    _excluded_cols[_CV_DATE_COL] = "cv_metadata"

    _rec_path = _namespace.merged_recommendations_path
    if _rec_path.exists():
        import yaml as _yaml

        from customer_retention.analysis.auto_explorer.layered_recommendations import RecommendationRegistry
        with _rec_path.open() as _f:
            _drop_recs = RecommendationRegistry.from_dict(_yaml.safe_load(_f))
        for _rec in getattr(getattr(_drop_recs, 'gold', None), 'feature_selection', []):
            if _rec.action in ('drop_multicollinear', 'drop_weak', 'drop_l1_zero', 'drop_availability', 'drop_zero_variance'):
                _excluded_cols[_rec.target_column] = _rec.action

    _exploration_profile = build_feature_profile("exploration", target, _profile_row_count, _profile_stats, _excluded_cols)
    _exploration_profile.save(_namespace.exploration_feature_profile_path)
    _n_sel_drops = sum(1 for v in _excluded_cols.values() if v.startswith("drop_"))
    print(f"Exploration feature profile saved: {_exploration_profile.feature_count} features, {_n_sel_drops} selection drops, {_profile_row_count:,} rows")

[//]: # (cr:doc name='8_3_baseline_models_with_class_weights' id=1aff1f67)
## 8.3 Baseline Models (with Class Weights)

**📖 Using Class Weights:**
- `class_weight='balanced'` automatically adjusts weights inversely proportional to class frequencies
- This helps models pay more attention to the minority class (churned customers)
- Without weights, models may just predict "retained" for everyone

In [ ]:
# @cr:code name='train_models' id=acf253e6
if not _skip_modeling:
    import time
    import warnings

    import numpy as np

    from customer_retention.stages.modeling import CrossValidator, CVStrategy

    if _use_distributed:
        from customer_retention.stages.modeling import create_distributed_models
        models = create_distributed_models(feature_names=_feature_names)
        print("Using distributed spark.ml models (data stays on cluster)")
    else:
        models = {
            "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
            "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'),
            "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
        }

    _avg = "binary" if _is_binary else "weighted"
    _cv_scoring = "roc_auc" if _is_binary else "f1_weighted"

    def _safe_auc(y_true, y_score, model_classes=None):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                if model_classes is None:
                    return roc_auc_score(y_true, y_score)
                return roc_auc_score(y_true, y_score, multi_class='ovr', labels=model_classes)
        except ValueError:
            return float('nan')

    # --- MLflow: parent run (same experiment as production) ---
    _mlflow_logger = MLflowLogger(experiment_name=_experiment_name, tracking_uri=_tracking_uri)
    _mlflow_logger.disable_autolog()
    _mlflow_logger.end_stale_runs()
    _mlflow_logger.start_run(run_name=f"exploration_{_cn}")
    _mlflow_logger.set_tags({
        "run_type": "exploration",
        "composite_name": _cn,
        "target_column": target,
        "entity_key": "entity_id",
        "timestamp_column": "as_of_date",
    })
    if _namespace:
        _mlflow_logger.set_tags({"cr_run_id": _namespace.run_id})
    _mlflow_logger.log_params({
        "train_samples": len(X_train) if not _use_distributed else train_count,
        "test_samples": len(X_test) if not _use_distributed else test_count,
        "n_features": len(_feature_names),
        "cv_folds": CV_FOLDS,
        "purge_gap_days": _purge_gap,
    })

    results = []
    model_predictions = {}

    for name, model in models.items():
        _t0 = time.monotonic()
        print(f"\nTraining {name}...")

        _use_scaled = "Logistic" in name
        _X_fit, _X_eval = (X_train_scaled, X_test_scaled) if _use_scaled else (X_train, X_test)

        with _mlflow_logger.nested_run(name):
            model.fit(_X_fit, y_train)
            _fit_elapsed = time.monotonic() - _t0
            print(f"  fit: {_fit_elapsed:.0f}s")

            y_pred_proba = model.predict_proba(_X_eval)
            y_pred = np.argmax(y_pred_proba, axis=1)

            if _is_binary:
                y_score = y_pred_proba[:, 1]
                auc = _safe_auc(y_test_np, y_score)
                pr_auc = average_precision_score(y_test_np, y_score)
            else:
                y_score = y_pred_proba
                auc = _safe_auc(y_test_np, y_score, model.classes_)
                pr_auc = float('nan')

            f1 = f1_score(y_test_np, y_pred, average=_avg, zero_division=0)
            precision = precision_score(y_test_np, y_pred, average=_avg, zero_division=0)
            recall = recall_score(y_test_np, y_pred, average=_avg, zero_division=0)

            def _on_fold(detail, fold_num, total_folds):
                print(f"  CV fold {fold_num}/{total_folds}: "
                      f"{_cv_scoring}={detail['score']:.4f} ({detail['elapsed_seconds']:.0f}s)")

            _cv = CrossValidator(strategy=CVStrategy.TEMPORAL_ENTITY, n_splits=CV_FOLDS, scoring=_cv_scoring, purge_gap_days=_purge_gap)
            _cv_result = _cv.run(model, _X_fit, y_train, groups=_train_entities, temporal_values=_train_dates, on_fold_complete=_on_fold)
            cv_scores = _cv_result.cv_scores

            _total = time.monotonic() - _t0
            print(f"  CV mean: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f} "
                  f"(total: {_total:.0f}s)")

            # Log metrics to MLflow nested run (same keys as production)
            _mlflow_logger.log_metrics({
                "roc_auc": auc, "pr_auc": pr_auc, "f1": f1,
                "precision": precision, "recall": recall,
                "cv_mean": float(cv_scores.mean()), "cv_std": float(cv_scores.std()),
            })

        results.append({
            "Model": name, "Test AUC": auc, "PR-AUC": pr_auc,
            "F1-Score": f1, "Precision": precision, "Recall": recall,
            "CV Score Mean": cv_scores.mean(), "CV Score Std": cv_scores.std()
        })

        model_predictions[name] = {'y_pred': y_pred, 'y_pred_proba': y_score}
        del y_pred_proba

    results_df = native_pd.DataFrame(results).round(4)

    _n_classes = len(_vc)
    print(f"\nCV method: temporal entity (GroupKFold + purge, {CV_FOLDS} folds)")
    print(f"CV metric: {'AUC' if _is_binary else 'F1-weighted'}")
    print(f"Classification type: {'binary' if _is_binary else f'multiclass ({_n_classes} classes)'}")
    print("\n" + "=" * 80)
    print("MODEL COMPARISON")
    print("=" * 80)
    display_table(results_df)

[//]: # (cr:doc name='8_4_feature_importance_random_forest' id=def19de4)
## 8.4 Feature Importance (Random Forest)

In [ ]:
# @cr:code name='plot_feature_importance' id=53277937
if not _skip_modeling:
    rf_model = models["Random Forest"]
    importance_df = native_pd.DataFrame({
        "Feature": _feature_names,
        "Importance": rf_model.feature_importances_
    }).sort_values("Importance", ascending=False)

    top_n = 15
    top_features = importance_df.head(top_n)

    fig = charts.bar_chart(
        top_features["Feature"].tolist(),
        top_features["Importance"].tolist(),
        title=f"Top {top_n} Feature Importances"
    )
    display_figure(fig)

    del models, X_train, X_test, X_train_scaled, X_test_scaled
    del y_train, _train_entities, _train_dates, _prep, _preparator


[//]: # (cr:doc name='8_5_classification_report_best_model' id=e6c13717)
## 8.5 Classification Report (Best Model)

In [ ]:
# @cr:code name='display_classification_report' id=ab0ef80e
if not _skip_modeling:
    print("Classification Report (Gradient Boosting):")
    print(classification_report(y_test_np, model_predictions["Gradient Boosting"]["y_pred"]))

[//]: # (cr:doc name='8_6_model_comparison_grid' id=4ece7d59)
## 8.6 Model Comparison Grid

This visualization shows all models side-by-side with:
- **Row 1**: Confusion matrices (counts and percentages)
- **Row 2**: ROC curves with AUC scores
- **Row 3**: Precision-Recall curves with PR-AUC scores

**📖 How to Read:**
- **Confusion Matrix**: Diagonal = correct predictions. Off-diagonal = errors.
- **ROC Curve**: Higher curve = better. AUC > 0.8 is good, > 0.9 is excellent.
- **PR Curve**: Higher curve = better at finding positives without false alarms.

In [ ]:
# @cr:code name='build_grid_results' id=4cc04512
if not _skip_modeling:
    grid_results = {
        name: {"y_pred": data["y_pred"], "y_pred_proba": data["y_pred_proba"]}
        for name, data in model_predictions.items()
    }

    if _is_binary:
        fig = charts.model_comparison_grid(
            grid_results, y_test_np,
            class_labels=["Churned (0)", "Retained (1)"],
            title="Model Comparison: Confusion Matrix | ROC Curve | Precision-Recall"
        )
        display_figure(fig)
    else:
        import numpy as np
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        from sklearn.metrics import confusion_matrix

        model_names = list(grid_results.keys())
        n_models = len(model_names)
        fig = make_subplots(rows=1, cols=n_models, subplot_titles=[f"{n[:20]}" for n in model_names])
        for i, name in enumerate(model_names):
            cm = confusion_matrix(y_test_np, grid_results[name]["y_pred"])
            fig.add_trace(go.Heatmap(
                z=cm, x=list(range(cm.shape[1])), y=list(range(cm.shape[0])),
                text=cm.astype(str), texttemplate="%{text}", showscale=False,
                colorscale="Blues",
            ), row=1, col=i + 1)
        fig.update_layout(title="Model Comparison: Confusion Matrices (multiclass)", height=400, width=350 * n_models + 50)
        display_figure(fig)

    print("\n" + "=" * 80)
    print("METRICS SUMMARY")
    print("=" * 80)
    _metrics_cols = ["Model", "Test AUC", "F1-Score", "Precision", "Recall"]
    if _is_binary:
        _metrics_cols.insert(2, "PR-AUC")
    display_table(results_df[_metrics_cols])

[//]: # (cr:doc name='8_6_1_individual_model_analysis' id=347acf32)
### 8.6.1 Individual Model Analysis

The grid above shows all models together. Below is detailed analysis per model.

In [ ]:
# @cr:code name='display_all_model_reports' id=c81d9e9c
if not _skip_modeling:
    print("=" * 70)
    print("CLASSIFICATION REPORTS BY MODEL")
    print("=" * 70)

    _target_names = ["Churned", "Retained"] if _is_binary else None

    for name, data in model_predictions.items():
        print(f"\n{'='*40}")
        print(f"  {name}")
        print('='*40)
        print(classification_report(y_test_np, data['y_pred'], target_names=_target_names, zero_division=0))
    del model_predictions


[//]: # (cr:doc name='8_6_1_precision_recall_curves' id=013c5047)
### 8.6.1 Precision-Recall Curves

**📖 Why PR Curves for Imbalanced Data:**
- ROC curves can look optimistic for imbalanced data
- PR curves focus on the minority class (churners)
- Better at showing how well we detect actual churners

**📖 How to Read:**
- **Baseline** (dashed line) = proportion of positives in the data
- Higher curve = better at finding churners without too many false alarms

[//]: # (cr:doc name='8_7_key_takeaways' id=ab825388)
## 8.7 Key Takeaways

**📖 Interpreting Results:**

In [ ]:
# @cr:code name='select_best_model' id=57fe85d1
if not _skip_modeling:
    _primary_metric = "Test AUC" if results_df["Test AUC"].notna().any() else "F1-Score"
    best_model = results_df.loc[results_df[_primary_metric].idxmax()]
    _best_name = best_model["Model"]
    _best_auc = best_model[_primary_metric]

    # --- MLflow: parent-level best metrics + features artifact ---
    _mlflow_logger.log_metrics({"best_roc_auc": float(_best_auc)})
    _mlflow_logger.set_tags({"best_model": _best_name})
    if _recs_hash:
        _mlflow_logger.set_tags({"recommendations_hash": _recs_hash})
    _mlflow_logger.log_dict({"feature_columns": _feature_names, "count": len(_feature_names)}, "features.json")
    _mlflow_run_id = _mlflow_logger.run_id  # capture before end_run clears it
    _mlflow_logger.end_run()

    # --- Persist exploration metadata to RunNamespace ---
    if _namespace:
        import json as _json
        _namespace.exploration_metadata_path.write_text(_json.dumps({
            "mlflow_experiment_name": _experiment_name,
            "mlflow_run_id": _mlflow_run_id,
            "composite_name": _cn,
            "target_column": target,
            "entity_key": "entity_id",
            "timestamp_column": "as_of_date",
            "recommendations_hash": _recs_hash or "",
            "best_model_name": _best_name,
            "best_roc_auc": float(_best_auc),
            "feature_columns": _feature_names,
            "run_type": "exploration",
        }))
        print(f"Exploration metadata saved to: {_namespace.exploration_metadata_path}")

    print("=" * 70)
    print("KEY TAKEAWAYS")
    print("=" * 70)

    print(f"\n  BEST MODEL (by {_primary_metric}): {_best_name}")
    if results_df["Test AUC"].notna().any():
        print(f"   Test AUC: {best_model['Test AUC']:.4f}")
    if _is_binary:
        print(f"   PR-AUC: {best_model['PR-AUC']:.4f}")
    print(f"   F1-Score: {best_model['F1-Score']:.4f}")

    print("\n  TOP 3 IMPORTANT FEATURES:")
    for i, row in enumerate(importance_df.head(3).itertuples(), 1):
        print(f"   {i}. {row.Feature} ({row.Importance:.3f})")

    _best_score = best_model[_primary_metric]
    print("\n  MODEL PERFORMANCE ASSESSMENT:")
    if _best_score > 0.90:
        print("   Excellent predictive signal - likely production-ready with tuning")
    elif _best_score > 0.80:
        print("   Strong predictive signal - good baseline for improvement")
    elif _best_score > 0.70:
        print("   Moderate signal - consider more feature engineering")
    else:
        print("   Weak signal - may need more data or different features")

    print("\n  NEXT STEPS:")
    print("   1. Feature engineering with derived features (notebook 05)")
    print("   2. Hyperparameter tuning (GridSearchCV)")
    print("   3. Threshold optimization for business metrics")
    print("   4. A/B testing in production")

    if MLFLOW_AVAILABLE:
        print(f"\n  MLflow experiment: {_experiment_name}")
        print("  MLflow run tagged: run_type=exploration")

[//]: # (cr:doc name='summary_what_we_learned' id=ad0dd3ec)
---

## Summary: What We Learned

In this notebook, we trained baseline models and established performance benchmarks:

1. **Data Preparation** - Proper train/test split with stratification and scaling
2. **Class Imbalance Handling** - Used balanced class weights
3. **Model Comparison** - Compared Logistic Regression, Random Forest, and Gradient Boosting
4. **Multiple Metrics** - Evaluated with AUC, PR-AUC, F1, Precision, Recall
5. **Feature Importance** - Identified the most predictive features

## Key Results for This Dataset

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Best AUC | ~0.98 | Excellent discrimination |
| Top Feature | esent | Email engagement is critical |
| Imbalance | ~4:1 | Moderate, handled with class weights |

---

## Next Steps

Continue to **09_business_alignment.ipynb** to:
- Align model performance with business objectives
- Define intervention strategies by risk level
- Calculate expected ROI from the model
- Set deployment requirements

In [ ]:
# @cr:code name='display_key_takeaways' id=a39b3757
if not _skip_modeling:
    _best_score_val = results_df[_primary_metric].max()

    print("Key Takeaways:")
    print("="*50)
    print(f"Best baseline {_primary_metric}: {_best_score_val:.4f}")
    print(f"Top 3 important features: {', '.join(importance_df.head(3)['Feature'].tolist())}")

    if _best_score_val > 0.85:
        print("\nStrong predictive signal detected. Data is well-suited for modeling.")
    elif _best_score_val > 0.70:
        print("\nModerate predictive signal. Consider feature engineering for improvement.")
    else:
        print("\nWeak predictive signal. May need more features or data.")

In [ ]:
# @cr:code name='release_stage_memory' id=c63e0281
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
del y_test_np, y_test, importance_df, results_df


[//]: # (cr:doc name='next_steps' id=627d05a3)
---

## Next Steps

Continue to **09_business_alignment.ipynb** to align with business objectives.

[//]: # (cr:doc name='section' id=3fda3e72)
> **Save Reminder:** Save this notebook (Ctrl+S / Cmd+S) before running the next one.
> The next notebook will automatically export this notebook's HTML documentation from the saved file.